```sh
 p3-all-genomes \     
  --eq genome_name,"Mycobacterium tuberculosis" \                          
  --eq genome_quality,"Good" \                                                                                         
  --eq antimicrobial_resistance,"Susceptible" \                                                            
  --attr antimicrobial_resistance,antimicrobial_resistance_evidence,collection_date,collection_year,common_name,genome_id,genome_name,genome_quality,geographic_group,geographic_location,host_common_name,phenotype,sra_accession > genomas_mtb_susceptibles.tbl
  ````

```sh
p3-get-genome-drugs \
  --in antibiotic,streptomycin,rifampin,pyrazinamide,isoniazid,ethambutol \
  -a antibiotic,computational_method,laboratory_typing_method,laboratory_typing_platform,resistant_phenotype,evidence \
  -i '/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/genomas_mtb_susceptibles_completos_obtener_perfil_resistencia.tbl' \
  -c 1 | tee drogas_filtrados_susceptibles.tbl
  ```

Obtener lista de genomas de MTB

In [1]:
import pandas as pd

bvbrc = pd.read_csv('/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/genomas_mtb_susceptibles.tbl', sep='\t')
# Renombrar columnas:
bvbrc.columns = bvbrc.columns.str.replace(r"^genome\.", "", regex=True)
bvbrc
# Obtener filas con SRA
bvbrc_sra_existentes = bvbrc[bvbrc['sra_accession'].notna()]
bvbrc_susceptibles = bvbrc_sra_existentes[bvbrc_sra_existentes['antimicrobial_resistance'] == "Susceptible"]
bvbrc_susceptibles_AMR_Panel = bvbrc_sra_existentes[bvbrc_sra_existentes['antimicrobial_resistance_evidence'] == "AMR Panel"]
# bvbrc_susceptibles_aleatorios = bvbrc_susceptibles_AMR_Panel.sample(n=1500, random_state=42) # Antes era 1000
bvbrc_susceptibles_sin_ers = bvbrc_susceptibles_aleatorios[~bvbrc_susceptibles_aleatorios['sra_accession'].str.startswith('ERS', na=False)]
bvbrc_susceptibles_sin_ers
bvbrc_susceptibles_sin_ers = (
    bvbrc[
        bvbrc['sra_accession'].notna()
        & (bvbrc['antimicrobial_resistance'] == "Susceptible")
        & (bvbrc['antimicrobial_resistance_evidence'] == "AMR Panel")
    ]
    .loc[lambda df: ~df['sra_accession'].str.startswith('ERS', na=False)]
    #.sample(n=1500, random_state=58)
)
bvbrc_susceptibles_sin_ers = bvbrc_susceptibles_sin_ers[~bvbrc_susceptibles_sin_ers["sra_accession"].str.contains(",")]
bvbrc_susceptibles_sin_ers.to_csv('/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/genomas_mtb_susceptibles_completos.tbl', index=False)
bvbrc_susceptibles_sin_ers


NameError: name 'bvbrc_susceptibles_aleatorios' is not defined

In [ ]:
df['geographic_location'].unique()

array([nan, 'Birmingham', 'Hamburg', 'Lima', 'Leeds', 'Guangdong',
       'Sichuan', 'Xinjiang', 'Jiangxi', 'Heilongjiang', 'Netherlands',
       'Norway', 'United Kingdom', 'Finland', 'Hunan', 'Henan', 'Tibet',
       'Shanxi', 'Guizhou', 'Guangxi', 'InnerMongolia',
       'Karolinska University Hospital', 'Tianjin', 'Shaanxi', 'Liaoning',
       'Yunnan', 'Anhui', 'Beijing', 'Hubei', 'Shandong', 'Hebei',
       'Chongqing', 'Zhejiang', 'Qinghai', 'Jiangsu', 'Jilin', 'Shanghai',
       'Hainan', 'Fujian', 'Gansu', 'Ningxia', 'Canada: British Columbia',
       'London', 'Italy', 'Bilthoven', 'Oxford', 'Valencia', 'Australia'],
      dtype=object)

In [ ]:
import pandas as pd
import numpy as np
df = bvbrc_susceptibles_sin_ers.copy() #pd.read_csv('/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/genomas_mtb_susceptibles_962.tbl')

df
df['geographic_location'].unique()
## Crar mapeo de regiones (Pais, Contienente)
mapa = {
    "Birmingham": ("United Kingdom", "Europe"),
    "Leeds": ("United Kingdom", "Europe"),
    "London": ("United Kingdom", "Europe"),
    "Oxford": ("United Kingdom", "Europe"),
    "United Kingdom": ("United Kingdom", "Europe"),

    "Hamburg": ("Germany", "Europe"),
    "Bilthoven": ("Netherlands", "Europe"),
    "Netherlands": ("Netherlands", "Europe"),
    "Norway": ("Norway", "Europe"),
    "Finland": ("Finland", "Europe"),
    "Italy": ("Italy", "Europe"),
    "Valencia": ("Spain", "Europe"),
    "Karolinska University Hospital": ("Sweden", "Europe"),

    "Australia": ("Australia", "Oceania"),

    "Canada: British Columbia": ("Canada", "North America"),

    "Lima": ("Peru", "South America"),

    "Guangdong": ("China", "Asia"),
    "Sichuan": ("China", "Asia"),
    "Xinjiang": ("China", "Asia"),
    "Jiangxi": ("China", "Asia"),
    "Heilongjiang": ("China", "Asia"),
    "Hunan": ("China", "Asia"),
    "Henan": ("China", "Asia"),
    "Tibet": ("China", "Asia"),
    "Shanxi": ("China", "Asia"),
    "Guizhou": ("China", "Asia"),
    "Guangxi": ("China", "Asia"),
    "InnerMongolia": ("China", "Asia"),
    "Tianjin": ("China", "Asia"),
    "Shaanxi": ("China", "Asia"),
    "Liaoning": ("China", "Asia"),
    "Yunnan": ("China", "Asia"),
    "Anhui": ("China", "Asia"),
    "Beijing": ("China", "Asia"),
    "Hubei": ("China", "Asia"),
    "Shandong": ("China", "Asia"),
    "Hebei": ("China", "Asia"),
    "Chongqing": ("China", "Asia"),
    "Zhejiang": ("China", "Asia"),
    "Qinghai": ("China", "Asia"),
    "Jiangsu": ("China", "Asia"),
    "Jilin": ("China", "Asia"),
    "Shanghai": ("China", "Asia"),
    "Hainan": ("China", "Asia"),
    "Fujian": ("China", "Asia"),
    "Gansu": ("China", "Asia"),
    "Ningxia": ("China", "Asia")
}

# Remapear las columnas
df['Pais'] = df["geographic_location"].map(lambda x: mapa.get(x, (np.nan, np.nan))[0])
df['geographic_group'] = df["geographic_location"].map(lambda x: mapa.get(x, (np.nan, np.nan))[1])
df = df[["genome_id", "sra_accession", "antimicrobial_resistance", "antimicrobial_resistance_evidence", "collection_year", "collection_date", "common_name", "genome_quality", "Pais", "geographic_group", "geographic_location", "phenotype", "host_common_name", "genome_name"]]
# df.to_csv("tablas/MTB_bvbrc_susceptibles_962.tbl",sep="\t", index=False)

In [ ]:
df = pd.read_csv("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/genomas_mtb_susceptibles_completos_obtener_perfil_resistencia.tbl", sep="\t")

In [ ]:
df

,genome_id,sra_accession,antimicrobial_resistance,antimicrobial_resistance_evidence,collection_year,collection_date,common_name,genome_quality,Pais,geographic_group,geographic_location,phenotype,host_common_name,genome_name
0,1773.27480,ERR552976,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_2566_11,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 2566-11
1,1773.28210,ERR551110,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_3305_05,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 3305-05
2,1773.30470,ERR552460,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_5489_06,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 5489-06
3,1773.30650,ERR551752,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_5661_08,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 5661-08
4,1773.33070,ERR551158,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_8023_12,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 8023-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8833,1773.16376,SRR7131294,Susceptible,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT76,Good,Italy,Europe,Italy,NaN,Human,Mycobacterium tuberculosis strain MGIT76
8834,1773.16377,SRR7131295,Susceptible,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT77,Good,Italy,Europe,Italy,NaN,Human,Mycobacterium tuberculosis strain MGIT77
8835,1773.16379,SRR7131297,Susceptible,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT79,Good,Italy,Europe,Italy,NaN,Human,Mycobacterium tuberculosis strain MGIT79
8836,1773.16383,SRR7131005,Susceptible,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT82,Good,Italy,Europe,Italy,NaN,Human,Mycobacterium tuberculosis strain MGIT82


### Ver las filas que ya estaban y añadir 500 mas

In [ ]:
# Ver si las filas orginales aun se conservan

df_origina = pd.read_csv("/Users/saitamawick98/Documents/💻/Susceptible/tablas/MTB_bvbrc_susceptibles_962.tbl", sep="\t") #usecols=["sra_accession"])
df_origina

# Entotal 961 genomas de MTB

sra_verificar = list(set(df_origina["sra_accession"].to_list()))

In [ ]:
df_500 = df[~df["sra_accession"].isin(sra_verificar)].sample(n=500, random_state=7)
df_500


,genome_id,sra_accession,antimicrobial_resistance,antimicrobial_resistance_evidence,collection_year,collection_date,common_name,genome_quality,Pais,geographic_group,geographic_location,phenotype,host_common_name,genome_name
4992,1773.20210,ERR245683,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR245683,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis ERR245683
6560,1773.48630,SRR2101082,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_C00016567,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis C00016567
7373,1733.66320,ERR2515925,Susceptible,AMR Panel,NaN,2012-2014,Mycobacterium_tuberculosis_BE01207626_S14,Good,NaN,NaN,NaN,NaN,Human,Mycobacterium tuberculosis BE01207626_S14
5582,1773.36120,ERR040098,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_C00003829,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis C00003829
2360,1733.48620,ERR2514147,Susceptible,AMR Panel,NaN,2009-2017,Mycobacterium_tuberculosis_13_0615042,Good,United Kingdom,Europe,Birmingham,NaN,Human,Mycobacterium tuberculosis 13.0615042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1133,1733.31940,ERR2512479,Susceptible,AMR Panel,NaN,2009-2017,Mycobacterium_tuberculosis_13_0610517,Good,United Kingdom,Europe,Birmingham,NaN,Human,Mycobacterium tuberculosis 13.0610517
6363,1773.45710,SRR2100780,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_C00014586,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis C00014586
4885,1773.20094,ERR216923,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR216923,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis ERR216923
5138,1773.20545,ERR2145503,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR2145503,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis ERR2145503


In [ ]:
# Agregar etiqueta de que ya existe 
import pandas as pd

In [ ]:
df_1 = pd.concat([df_origina, df_500], keys=["ya hay info", "obtener info"]).reset_index(drop=True, level=1).reset_index()

In [ ]:
# Buscar archivos de TBProfiler

df_1.to_csv('/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/obtener_perfil_de_resistencia_bvbrc.tbl', sep="\t", index=False)

# Aqui comienza

In [2]:
import pandas as pd

df_drogas = pd.read_csv("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/drogas_filtrados_susceptibles.tbl", sep='\t')
df_drogas

df_metadatos = pd.read_csv("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/drogas_filtrados_susceptibles.tbl", sep='\t', usecols=["genome_id", "sra_accession", "antimicrobial_resistance_evidence", "collection_year", "collection_date", "common_name", "genome_quality", "Pais", "geographic_group", "genome_name"])

/var/folders/8s/0zjwys_9585g_ct_14ccvp9r0000gn/T/ipykernel_78609/3490457492.py:3: DtypeWarning: Columns (5,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_drogas = pd.read_csv("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/drogas_filtrados_susceptibles.tbl", sep='\t')
/var/folders/8s/0zjwys_9585g_ct_14ccvp9r0000gn/T/ipykernel_78609/3490457492.py:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_metadatos = pd.read_csv("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/drogas_filtrados_susceptibles.tbl", sep='\t', usecols=["genome_id", "sra_accession", "antimicrobial_resistance_evidence", "collection_year", "collection_date", "common_name", "genome_quality", "Pais", "geographic_group", "genome_name"])


In [3]:
df_drogas

,genome_id,sra_accession,antimicrobial_resistance,antimicrobial_resistance_evidence,collection_year,collection_date,common_name,genome_quality,Pais,geographic_group,geographic_location,phenotype,host_common_name,genome_name,genome_drug.antibiotic,genome_drug.computational_method,genome_drug.laboratory_typing_method,genome_drug.laboratory_typing_platform,genome_drug.resistant_phenotype,genome_drug.evidence
0,1773.27480,ERR552976,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_2566_11,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 2566-11,pyrazinamide,NaN,NaN,NaN,Susceptible,Laboratory Method
1,1773.27480,ERR552976,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_2566_11,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 2566-11,pyrazinamide,MIC XGBoost Model,NaN,NaN,NaN,Computational Method
2,1773.27480,ERR552976,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_2566_11,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 2566-11,pyrazinamide,SIR XGBoost Model,NaN,NaN,Susceptible,Computational Method
3,1773.27480,ERR552976,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_2566_11,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 2566-11,ethambutol,NaN,NaN,NaN,Susceptible,Laboratory Method
4,1773.27480,ERR552976,Susceptible,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_2566_11,Good,NaN,NaN,NaN,NaN,NaN,Mycobacterium tuberculosis 2566-11,ethambutol,MIC XGBoost Model,NaN,NaN,NaN,Computational Method
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142392,1773.18095,SRS5709778,Susceptible,AMR Panel,2017.0,2017,Mycobacterium_tuberculosis,Good,Australia,Oceania,Australia,NaN,Human,Mycobacterium tuberculosis strain AUSMDU00018547,rifampin,MIC XGBoost Model,NaN,NaN,NaN,Computational Method
142393,1773.18095,SRS5709778,Susceptible,AMR Panel,2017.0,2017,Mycobacterium_tuberculosis,Good,Australia,Oceania,Australia,NaN,Human,Mycobacterium tuberculosis strain AUSMDU00018547,rifampin,SIR XGBoost Model,NaN,NaN,Susceptible,Computational Method
142394,1773.18095,SRS5709778,Susceptible,AMR Panel,2017.0,2017,Mycobacterium_tuberculosis,Good,Australia,Oceania,Australia,NaN,Human,Mycobacterium tuberculosis strain AUSMDU00018547,streptomycin,AdaBoost Classifier,NaN,NaN,Susceptible,Computational Method
142395,1773.18095,SRS5709778,Susceptible,AMR Panel,2017.0,2017,Mycobacterium_tuberculosis,Good,Australia,Oceania,Australia,NaN,Human,Mycobacterium tuberculosis strain AUSMDU00018547,streptomycin,MIC XGBoost Model,NaN,NaN,NaN,Computational Method


In [4]:
# Filtrar los datos que no existen en drug_resistant

df_drogas.columns = df_drogas.columns.str.removeprefix("genome_drug.")

In [5]:
# Mantener filas que contienen informacion de resistencia
df_drogas = df_drogas[df_drogas["resistant_phenotype"].notna()]

# Renombrar valores
df_drogas = df_drogas.replace({
    "antibiotic": {
        "pyrazinamide" : "PZA",
        "ethambutol" : "EMB",
        "isoniazid" : "INH",
        "streptomycin" : "STR",
        "rifampin" : "RIF"
    },
    "resistant_phenotype" : {
        "Susceptible" : "S",
        "Resistant" : "R"
    },
    "evidence" : {
        "Laboratory Method" : "Lab",
        "Computational Method" : "Comp" 
    }
}).pivot_table(
    values="resistant_phenotype",
    index=["sra_accession", "antibiotic"],
    columns="evidence",
    aggfunc=lambda x: ",".join(set(x)),
).reset_index().dropna()

In [6]:
df_drogas

evidence,sra_accession,antibiotic,Comp,Lab
0,ERR036186,EMB,S,S
1,ERR036186,INH,S,S
2,ERR036186,PZA,S,S
3,ERR036186,RIF,S,S
4,ERR036186,STR,"R,S",S
...,...,...,...,...
43298,SRR7131297,RIF,S,S
43300,SRS5709778,EMB,S,S
43301,SRS5709778,INH,S,S
43302,SRS5709778,PZA,S,S


In [7]:
def evaluar_inconsistencias(comp, lab):
    if comp in ["R", "S"] and lab in ["R", "S"] :
        return comp != lab
    else:
        return False

df_drogas["inconsistente"] = df_drogas.apply(
    lambda x: evaluar_inconsistencias(x["Comp"], x["Lab"]),
    axis=1
)

df_drogas



evidence,sra_accession,antibiotic,Comp,Lab,inconsistente
0,ERR036186,EMB,S,S,False
1,ERR036186,INH,S,S,False
2,ERR036186,PZA,S,S,False
3,ERR036186,RIF,S,S,False
4,ERR036186,STR,"R,S",S,False
...,...,...,...,...,...
43298,SRR7131297,RIF,S,S,False
43300,SRS5709778,EMB,S,S,False
43301,SRS5709778,INH,S,S,False
43302,SRS5709778,PZA,S,S,False


In [8]:
# Mantener los genomas que contienen los cinco antibioticos:

antibioticos_esperados = {"EMB", "INH", "PZA", "RIF", "STR"}

conteo = df_drogas.groupby("sra_accession")["antibiotic"].nunique()

genomas_completos = conteo[conteo == 5].index

df_5_ab = df_drogas[df_drogas["sra_accession"].isin(genomas_completos)]

In [9]:
ab_completos =(df_5_ab[df_5_ab["inconsistente"] == False]
    .groupby("sra_accession")["antibiotic"]
    .nunique()
    .loc[lambda x: x == 5].index)

df_5_ab[df_5_ab["sra_accession"].isin(ab_completos)]

evidence,sra_accession,antibiotic,Comp,Lab,inconsistente
0,ERR036186,EMB,S,S,False
1,ERR036186,INH,S,S,False
2,ERR036186,PZA,S,S,False
3,ERR036186,RIF,S,S,False
4,ERR036186,STR,"R,S",S,False
...,...,...,...,...,...
43165,SRR7131135,EMB,R,R,False
43166,SRR7131135,INH,R,R,False
43167,SRR7131135,PZA,R,R,False
43168,SRR7131135,RIF,R,R,False


In [10]:
df_5_ab = df_5_ab[df_5_ab["sra_accession"].isin(ab_completos)]
df_5_ab

evidence,sra_accession,antibiotic,Comp,Lab,inconsistente
0,ERR036186,EMB,S,S,False
1,ERR036186,INH,S,S,False
2,ERR036186,PZA,S,S,False
3,ERR036186,RIF,S,S,False
4,ERR036186,STR,"R,S",S,False
...,...,...,...,...,...
43165,SRR7131135,EMB,R,R,False
43166,SRR7131135,INH,R,R,False
43167,SRR7131135,PZA,R,R,False
43168,SRR7131135,RIF,R,R,False


In [11]:
df_5_ab = df_5_ab.merge(df_metadatos.drop_duplicates(), on="sra_accession", how="left")
df_5_ab


,sra_accession,antibiotic,Comp,Lab,inconsistente,genome_id,antimicrobial_resistance_evidence,collection_year,collection_date,common_name,genome_quality,Pais,geographic_group,genome_name
0,ERR036186,EMB,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
1,ERR036186,INH,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
2,ERR036186,PZA,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
3,ERR036186,RIF,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
4,ERR036186,STR,"R,S",S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8030,SRR7131135,EMB,R,R,False,1773.16340,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT36,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT36
8031,SRR7131135,INH,R,R,False,1773.16340,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT36,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT36
8032,SRR7131135,PZA,R,R,False,1773.16340,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT36,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT36
8033,SRR7131135,RIF,R,R,False,1773.16340,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT36,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT36


In [12]:
genomas_S5 = (
    df_5_ab.groupby("sra_accession")
    .filter(lambda x: (x["Lab"] == "S").all() and len(x) == 5)
)

genomas_S5

,sra_accession,antibiotic,Comp,Lab,inconsistente,genome_id,antimicrobial_resistance_evidence,collection_year,collection_date,common_name,genome_quality,Pais,geographic_group,genome_name
0,ERR036186,EMB,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
1,ERR036186,INH,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
2,ERR036186,PZA,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
3,ERR036186,RIF,S,S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
4,ERR036186,STR,"R,S",S,False,1773.18718,AMR Panel,NaN,NaN,Mycobacterium_tuberculosis_ERR036186,Good,NaN,NaN,Mycobacterium tuberculosis ERR036186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8025,SRR7131127,EMB,S,S,False,1773.16230,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT189,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT189
8026,SRR7131127,INH,S,S,False,1773.16230,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT189,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT189
8027,SRR7131127,PZA,S,S,False,1773.16230,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT189,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT189
8028,SRR7131127,RIF,S,S,False,1773.16230,AMR Panel,NaN,2016-2017,Mycobacterium_tuberculosis_strain_MGIT189,Good,Italy,Europe,Mycobacterium tuberculosis strain MGIT189


In [13]:
sra_list = (
    df_5_ab[df_5_ab["inconsistente"] == False]
    .groupby("sra_accession")["antibiotic"]
    .nunique()
    .loc[lambda x: x == 5]
    .index
    .to_list()
)

with open("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/sra_ids_susceptibles.txt", "w") as f:
    f.write("SRA\n" + "\n".join(sra_list))

In [14]:
from pathlib import Path

ruta_sus = Path("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/snippy/sift_susceptibles/resultados_sift")

files_sus_xls = [x.name.split("_")[0].strip() for x in ruta_sus.iterdir() if x.name.endswith(".xls")]


In [15]:
df_sus = genomas_S5.loc[genomas_S5["sra_accession"].isin(files_sus_xls)]



Juntar TBprofiler con los metadatos

In [16]:
tbp_df = pd.read_csv("/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/TBProfiler/TBProfiler_67_samples.csv")
tbp_df

# Remplazar valores

tbp_df["Linaje"] = tbp_df["Linaje"].str.replace("lineage", "L", regex=False)
tbp_df["Sub_linaje"] = tbp_df["Sub_linaje"].str.replace("lineage", "L", regex=False)
tbp_df["Resultado"] = tbp_df["Resultado"].str.replace("No detectado (Sensible)", "S", regex=False)

In [17]:
df_listo = tbp_df[tbp_df["id"].isin(files_sus_xls)].merge(df_sus.drop_duplicates(), left_on=["id", "Antibiotico_Reporte"], right_on=["sra_accession", "antibiotic"], how="left")

In [18]:
ids_faltantes_metadatos = df_listo[df_listo["sra_accession"].isna()]["id"].drop_duplicates().to_list()

len(ids_faltantes_metadatos)

21

In [19]:
faltantes_21 = df_5_ab[df_5_ab["sra_accession"].isin(ids_faltantes_metadatos)]

In [20]:
un = df_listo.merge(faltantes_21, left_on=["id", "Antibiotico_Reporte"], right_on=["sra_accession", "antibiotic"], how="left")

In [21]:
un["sra_accession_x"] = un["sra_accession_x"].fillna(un["sra_accession_y"])
un.drop(columns="sra_accession_y", inplace=True)

un["antibiotic_x"] = un["antibiotic_x"].fillna(un["antibiotic_y"])
un.drop(columns="antibiotic_y", inplace=True)

un["Comp_x"] = un["Comp_x"].fillna(un["Comp_y"])
un.drop(columns="Comp_y", inplace=True)

un["Lab_x"] = un["Lab_x"].fillna(un["Lab_y"])
un.drop(columns="Lab_y", inplace=True)

un["inconsistente_x"] = un["inconsistente_x"].fillna(un["inconsistente_y"])
un.drop(columns="inconsistente_y", inplace=True)

un["genome_id_x"] = un["genome_id_x"].fillna(un["genome_id_y"])
un.drop(columns="genome_id_y", inplace=True)

un["antimicrobial_resistance_evidence_x"] = un["antimicrobial_resistance_evidence_x"].fillna(un["antimicrobial_resistance_evidence_y"])
un.drop(columns="antimicrobial_resistance_evidence_y", inplace=True)

un["collection_year_x"] = un["collection_year_x"].fillna(un["collection_year_y"])
un.drop(columns="collection_year_y", inplace=True)

un["collection_date_x"] = un["collection_date_x"].fillna(un["collection_date_y"])
un.drop(columns="collection_date_y", inplace=True)

un["common_name_x"] = un["common_name_x"].fillna(un["common_name_y"])
un.drop(columns="common_name_y", inplace=True)


un["genome_quality_x"] = un["genome_quality_x"].fillna(un["genome_quality_y"])
un.drop(columns="genome_quality_y", inplace=True)


un["Pais_x"] = un["Pais_x"].fillna(un["Pais_y"])
un.drop(columns="Pais_y", inplace=True)


un["geographic_group_x"] = un["geographic_group_x"].fillna(un["geographic_group_y"])
un.drop(columns="geographic_group_y", inplace=True)


un["genome_name_x"] = un["genome_name_x"].fillna(un["genome_name_y"])
un.drop(columns="genome_name_y", inplace=True)


/var/folders/8s/0zjwys_9585g_ct_14ccvp9r0000gn/T/ipykernel_78609/20097920.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  un["inconsistente_x"] = un["inconsistente_x"].fillna(un["inconsistente_y"])


In [22]:
un.to_csv('/Users/saitamawick98/Documents/💻/Febrero 2026/Susceptible/df_susceptible.csv', index=False)